In [5]:

text_maintenance =  """معاملة 12600176, واردة, الجهة المرسلة undefined إدارة (-), الجهة المستقبلة مدينة الملك عبدالعزيز للعلوم والتقنية للذكاء الاصطناعي إدارة (المكتب التنفيذي), بتاريخ 2022-10-25 م / 1448-04-06 ه, بشأن حوكمة التنسيق بين وزارة الصناعة والثروة المعدنية--- Page 1 --- بسم الله الرحمن الرحيم
المملكة العربية السعودية الأمانة العامة لمجلس الوزراء قرار مجلس الوزراء
قرار رقم: (٢٣٥) وتاريخ: ٢٩ / ٣ / ١٤٤٤ه
إن مجلس الوزراء بعد الاطلاع على المعاملة الواردة من الديوان الملكي برقم ٧٣٦٢١ وتاريخ ٢٤ / ١١ / ١٤٤٣ه، المشتملة على برقية معالي وزير الصناعة والثروة المعدنية رقم ٢٩٨٥ وتاريخ ١٨ / ٤ / ١٤٤١ه، في شأن مشروع حوكمة التنسيق بين وزارة الصناعة والثروة المعدنية والجهات المختصة في شأن الإجراءات المتعلقة بالمنشآت الصناعية. وبعد الاطلاع على مشروع الحوكمة المشار إليه. وبعد الاطلاع على الأمر الملكي رقم (٣٤١٤) وتاريخ ١٨ / ١ / ١٤٤١ه. وبعد الاطلاع على المحضرين رقم (٣٢٩) وتاريخ ٢٧ / ٥ / ١٤٤٢ه، ورقم (٣٩٩) وتاريخ ٤ / ٩ / ١٤٤٣ه، والمذكرة رقم (٦٤٣) وتاريخ ٢٩ / ٢ / ١٤٤٤ه، المعدة في هيئة الخبراء بمجلس الوزراء. وبعد الاطلاع على التوصية المعدة في مجلس الشؤون الاقتصادية والتنمية رقم (١-٤٤/١/د) وتاريخ ٦ / ١ / ١٤٤٤ه. وبعد الاطلاع على توصية اللجنة العامة لمجلس الوزراء رقم (٢٥٥٧) وتاريخ ١٣ / ٣ / ١٤٤٤ه.
يقرر ما يلي:
أولا: الموافقة على حوكمة التنسيق بين وزارة الصناعة والثروة المعدنية والجهات المختصة في شأن الإجراءات المتعلقة بالمنشآت الصناعية، بالصيغة المرافقة.
ثانيا: قيام وزارة الصناعة والثروة المعدنية بالآتي:
١- إنشاء منصة إلكترونية للقيام بكل ما يلزم في سبيل تيسير تطبيق الأحكام الواردة في الحوكمة المشار إليها في البند (أولا) من هذا القرار، دون إخلال باختصاصات ومسؤوليات الجهات الأخرى، ووفقا لأفضل الممارسات الدولية.
--- Page 2 --- بسم الله الرحمن الرحيم
المملكة العربية السعودية الأمانة العامة لمجلس الوزراء قرارات مجلس الوزراء
(٢)
٢- إعداد تقرير تقويم للحوكمة - المشار إليها في البند (أولا) من هذا القرار- بعد مرور (سنة) من تاريخ الموافقة عليها، ورفعه إلى مجلس الشؤون الاقتصادية والتنمية، لاتخاذ ما يراه."""


text_kabsa = "هذا النص عبارة عن خرابيط"

#bge-m3@1024

In [3]:
import requests
import numpy as np

URL = "http://34.196.113.253:8000/v1/ai/embed"
HEADERS = {
    "accept": "application/json",
    "X-API-Key": "123",
    "Content-Type": "application/json"
}

def get_embedding(text):
    resp = requests.post(URL, headers=HEADERS, json={
        "texts": [text],
        "format": "float32"
    })
    resp.raise_for_status()
    return resp.json()["vectors"][0]

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

emb1 = get_embedding(text_maintenance)
emb2 = get_embedding(text_kabsa)

raw_cosine = cosine_sim(emb1, emb2)
opensearch_score = (1 + raw_cosine) / 2

print(f"Raw cosine similarity: {raw_cosine:.4f}")
print(f"OpenSearch score equivalent: {opensearch_score:.4f}")

Raw cosine similarity: 0.3655
OpenSearch score equivalent: 0.6828


# gemini-embedding-001

In [4]:
!pip install google-genai

In [6]:
from google import genai
from google.genai import types
import numpy as np

client = genai.Client(api_key="API_KEY")

def get_embedding(text, task_type="RETRIEVAL_DOCUMENT"):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=types.EmbedContentConfig(
            task_type=task_type,   # مهم! فرّق بين query و document
            output_dimensionality=1024  # نفس أبعاد BGE عندك للمقارنة العادلة
        )
    )
    return result.embeddings[0].values

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


emb1 = get_embedding(text_maintenance, task_type="RETRIEVAL_DOCUMENT")
emb2 = get_embedding(text_kabsa, task_type="RETRIEVAL_DOCUMENT")

raw_cosine = cosine_sim(emb1, emb2)
opensearch_score = (1 + raw_cosine) / 2

print(f"Raw cosine similarity: {raw_cosine:.4f}")
print(f"OpenSearch score equivalent: {opensearch_score:.4f}")

Raw cosine similarity: 0.6839
OpenSearch score equivalent: 0.8419
